# Data Generation

This notebook simulates pp collisions at the STAR experiment and stores them into a ROOT tree. Afterwards, the data are smeared to simulate detector effects.

### ROOT Tree Generation Using PYTHIA8

At first, it is necessary to import ROOT, PYTHIA8, and NumPy.

In [1]:
import ROOT
import pythia8
import numpy as np

ROOT.EnableImplicitMT() # enable multi-threading

/home/javkovsky/anaconda3/envs/star-bachelor-env/lib/python3.11/site-packages/cppyy/__init__.py:374: UserWarning: CPyCppyy API not found (tried: /home/javkovsky/anaconda3/envs/star-bachelor-env/include/site/python3.11); set CPPYY_API_PATH envar to the 'CPyCppyy' API directory to fix
  warnings.warn("CPyCppyy API not found (tried: %s); "


Then, we configure PYTHIA8 for $\sqrt{s_\mathrm{NN}} = 510 \: \mathrm{GeV}$ pp collisions and use the Detroit tune for RHIC.

In [2]:
pythia = pythia8.Pythia()

pythia.readString("Beams:idA = 2212") # proton
pythia.readString("Beams:idB = 2212") # proton
pythia.readString("Beams:eCM = 510.") # collision energy in GeV

pythia.readString("hardQCD:all = on") # enable hard QCD processes
pythia.readString("PhaseSpace:pTHatMin = 20.") # minimum pT of scattered partons in GeV

pythia.readString("PartonLevel:MPI = on") # enable multi-parton interactions
pythia.readString("PartonLevel:ISR = on") # enable initial state radiation
pythia.readString("PartonLevel:FSR = on") # enable final state radiation
pythia.readString("HadronLevel:Decay = on") # enable decay of hadronization products

pythia.readString("Tune:pp = 33") # the Detroit tune for RHIC

seed = 420 # choose a specific seed
pythia.readString("Random:setSeed = on") # enable random seed setting
pythia.readString(f"Random:seed = {seed}") # set random seed to 0 (random every time) or to a specific value for reproducibility

pythia.init() # initialize PYTHIA8

True


 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.312      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 23 May 2024  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T    H   H  III  A   A    Now is 18 Apr 2026 at 11:04:15    |  | 
 |  |                                                                              |  | 
 |  |   Program docu

We prepare the ROOT tree with branches which we need to fill later with event data.

In [3]:
file = ROOT.TFile(f"data/events{seed}.root", "RECREATE") # create a .root file
tree = ROOT.TTree("events", "A tree with pp events from PYTHIA8") # create the tree

# ALLOCATE MEMORY FOR OBSERVABLES
# scalars
eventID = np.zeros(1, dtype='u8') # event ID (unsigned 64-bit integer = 'unsigned long long' data type in C++)

# vectors
pTVec = ROOT.std.vector('double')() # pT (double precision)
etaVec = ROOT.std.vector('double')() # eta (double precision)
phiVec = ROOT.std.vector('double')() # phi (double precision)
massVec = ROOT.std.vector('double')() # mass (double precision)

# CREATE BRANCHES
tree.Branch("eventID", eventID, "eventID/l") # "/l" specifies an unsigned 64-bit integer

tree.Branch("pT", pTVec)
tree.Branch("eta", etaVec)
tree.Branch("phi", phiVec)
tree.Branch("mass", massVec)

Afterwards, we generate the events.

In [4]:
nEvents = 100000 # set the number of events
for iEvent in range(nEvents): # event loop
    
    # clear objects where we store observables for each event
    pTVec.clear()
    etaVec.clear()
    phiVec.clear()
    massVec.clear()

    if not pythia.next(): # generate an event
        continue

    # store the event ID
    eventID[0] = iEvent

    for i in range(pythia.event.size()): # particle loop
        particle = pythia.event[i]
        if particle.isFinal() and particle.isCharged() and abs(particle.eta())<1 and abs(particle.pT())>0.2: # select only final state charged particles in midrapidity, add pT cut of 0.2 GeV typical for STAR
            # fill the observable storage with respective data
            pTVec.push_back(particle.pT())
            etaVec.push_back(particle.eta())
            phiVec.push_back(particle.phi())
            massVec.push_back(particle.m())

    tree.Fill()


 --------  PYTHIA Info Listing  ---------------------------------------- 
 
 Beam A: id =   2212, pz =  2.550e+02, e =  2.550e+02, m =  9.383e-01.
 Beam B: id =   2212, pz = -2.550e+02, e =  2.550e+02, m =  9.383e-01.

 In 1: id =   21, x =  1.468e-01, pdf =  6.335e-01 at Q2 =  4.346e+02.
 In 2: id =    2, x =  7.257e-02, pdf =  6.161e-01 at same Q2.

 Subprocess q g -> q g with code 113 is 2 -> 2.
 It has sHat =  2.770e+03,    tHat = -5.397e+02,    uHat = -2.230e+03,
       pTHat =  2.085e+01,   m3Hat =  0.000e+00,   m4Hat =  3.300e-01,
    thetaHat =  9.143e-01,  phiHat =  2.354e+00.
     alphaEM =  7.689e-03,  alphaS =  1.697e-01    at Q2 =  4.346e+02.

 Impact parameter b =  4.241e-01 gives enhancement factor =  2.402e+00.
 Max pT scale for MPI =  2.085e+01, ISR =  2.085e+01, FSR =  2.085e+01.
 Number of MPI =     7, ISR =     9, FSRproc =    35, FSRreson =     0.

 --------  End PYTHIA Info Listing  ------------------------------------

 --------  PYTHIA Event Listing  (hard proc

Finally, we save everything into the .root file and close it.

In [5]:
tree.Write()
file.Close()

### Simulating Detector Effects

Now, we have the simulated particle collisions from PYTHIA8 (the "true" data) and would like to obtain a data set similar to the ones we measure at STAR. In order to do so, we must simulate the detector effects of the STAR detector.

We are using RDataFrame for work with the data. We define a new data frame containing 4-vectors (tracks) of $p_\mathrm{T}$, $\eta$, $\phi$, mass using LorentzVector (PtEtaPhiMVector).

In [6]:
import ROOT # tyto 2 radky smazat nakonec
ROOT.EnableImplicitMT()

df = ROOT.RDataFrame("events", f"data/events{seed}.root") # connect RDataFrame to the TTree in events{seed}.root

dfTracks = df.Define("tracks", "ROOT::VecOps::Construct<ROOT::Math::PtEtaPhiMVector> (pT, eta, phi, mass)") # create tracks -- put pT, eta, phi, mass into a LorentzVector

!!! SO FAR, THE RESOLUTION AND ACCEPTANCE FUNCTIONS ARE BASED FOR PIONS -- I NEED TO ADD OTHER PARTICLES IN THE FUTURE !!!

Firstly, we simulate the **acceptance of TPC and TOF**.

In [7]:
# I wrote a function TPCacceptance(Nch, pT) which randomly selects particles in an event based on an experimentally determined TPCs acceptance function (!!! now it's constant for all pT values, but will be modified to match experiments in the future) and a function TPCandTOFacceptance(Nch, pT, TPCaccepted) which works similarly. We need to tell ROOT to load and compile them.
ROOT.gInterpreter.ProcessLine('#include "cpp/TPCandTOFacceptance.cpp"')

dfAccepted = (
    dfTracks.Define("TPCaccepted", "TPCmask(tracks.size(), pT, eventID)") # define a new column with events accepted (1) and rejected (0) by TPC
             .Define("TPCandTOFaccepted", "TPCandTOFmask(tracks.size(), pT, eventID, TPCaccepted)") # define a new column with events accepted (1) and rejected (0) by both TPC and TOF
             .Define("acceptedTracks", "tracks[TPCandTOFaccepted]") # define a new column to store only accepted tracks
)

Next, we need to smear $p_\mathrm{T}$, $\phi$, and $\eta$ to simulate **imperfect resolution of the detector**.

In [8]:
ROOT.gInterpreter.ProcessLine('#include "cpp/smearing.cpp"') # load C++ function which smears the accepted tracks
ROOT.gInterpreter.ProcessLine('#include "cpp/extract_components.cpp"') # load C++ functions which extract components of our tracks

dfSmeared = (
    dfAccepted.Define("smearedTracks", "smearing(acceptedTracks, eventID)")
               .Define("pTSmeared", "getpT(smearedTracks)")
               .Define("etaSmeared", "getEta(smearedTracks)")
               .Define("phiSmeared", "getPhi(smearedTracks)")
)

In the end, we extract each 4-vector component into columns and save these columns into a new tree composed of smeared data.

In [9]:
# extract smeared tracks
# df_smeared = (
#     df_accepted.Redefine("pT", "getpT(acceptedTracks)")
#                .Redefine("eta", "getEta(acceptedTracks)")
#                .Redefine("phi", "getPhi(acceptedTracks)")
#                .Redefine("mass", "getMass(acceptedTracks)")
# )

# save the smeared tree into a .root file
opts = ROOT.RDF.RSnapshotOptions() # ensure that events_smeared.root file is recreated every time this notebook is ran
opts.fMode = "RECREATE"
dfSmeared.Snapshot("events_smeared", f"data/events{seed}_smeared.root", ['eventID', 'pT', 'eta', 'phi', 'mass', 'TPCaccepted', 'TPCandTOFaccepted', 'pTSmeared', 'etaSmeared', 'phiSmeared'], opts)

<cppyy.gbl.ROOT.RDF.RResultPtr<ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager,void> > object at 0x562dd07ff880>

Info in <[ROOT.RDF] Info /home/conda/feedstock_root/build_artifacts/bld/rattler-build_root_base_1771471359/work/build-dir/include/ROOT/RDF/RInterface.hxx:1363 in auto ROOT::RDF::RInterface<ROOT::Detail::RDF::RLoopManager, void>::Snapshot(std::string_view, std::string_view, const ColumnNames_t &, const RSnapshotOptions &)::(anonymous class)::operator()() const [Proxied = ROOT::Detail::RDF::RLoopManager, DataSource = void]>: 
	In ROOT 6.38, the default compression settings of Snapshot have been changed from 101 (ZLIB with compression level 1, the TTree default) to 505 (ZSTD with compression level 5). This change may result in smaller Snapshot output dataset size by default. In order to suppress this message, set 'ROOT_RDF_SNAPSHOT_INFO=0' in your environment or set 'ROOT.RDF.Snapshot.Info: 0' in your .rootrc file.
